# Cardiac MRI segmentation with U-Net

This notebook records the final Group 30 experiment for four-class cardiac MRI segmentation. The training and evaluation outputs are retained from the coursework run; the dataset and saved checkpoint are not distributed with this repository.


## Setup


In [ ]:
import os

import cv2
import torch
from torch import nn, optim
from torch.utils.data import DataLoader

from unet_cardiac_segmentation import (
    SegmentationDataset,
    TestImageDataset,
    UNet,
    train_epoch,
    validate_epoch,
)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"The current device is {device}")

## Data


In [ ]:
training_data_path = "./data/train"
validation_data_path = "./data/val"
test_data_path = "./data/test"

num_workers = 4
batch_size = 4

training_set = SegmentationDataset(training_data_path)
training_data_loader = DataLoader(
    dataset=training_set,
    num_workers=num_workers,
    batch_size=batch_size,
    shuffle=True,
)

validation_set = SegmentationDataset(validation_data_path)
validation_data_loader = DataLoader(
    dataset=validation_set,
    num_workers=num_workers,
    batch_size=batch_size,
    shuffle=True,
)

test_set = TestImageDataset(test_data_path)
test_data_loader = DataLoader(
    dataset=test_set,
    num_workers=num_workers,
    batch_size=batch_size,
    shuffle=False,
)

## Model and optimiser

The model uses four downsampling stages, four upsampling stages with skip connections, and a four-channel output projection.


In [ ]:
model = UNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    amsgrad=True,
    weight_decay=1e-8,
)

## Training

The output below is the complete log retained from the recorded 100-epoch run.


In [9]:
epochs = 100
max_dice_score = 0.0

for epoch in range(epochs):
    training_loss = train_epoch(
        model,
        training_data_loader,
        optimizer,
        criterion,
        device,
    )
    validation_loss, score = validate_epoch(
        model,
        validation_data_loader,
        criterion,
        device,
    )
    validation_dice = score / len(validation_data_loader)

    mean_training_loss = training_loss / len(training_data_loader)
    mean_validation_loss = validation_loss / len(validation_data_loader)
    print(
        f"Epoch: {epoch + 1}, Training Loss: {mean_training_loss:.6f}, "
        f"Valid Loss: {mean_validation_loss:.6f}, Dice Score: {validation_dice:.6f}"
    )

    if max_dice_score < validation_dice:
        print(f"Dice Score Increased {max_dice_score:.6f} -> {validation_dice:.6f}")
        max_dice_score = validation_dice
        print("Saving the model")
        torch.save(model.state_dict(), "saved_model.pth")

Epoch: 1,           Training Loss: 1.020634,           Valid Loss: 31.709321,           Dice Score: 0.046423
Dice Score Increased 0.000000 -> 0.046423
Saving the model
Epoch: 2,           Training Loss: 0.503054,           Valid Loss: 0.830373,           Dice Score: 0.444213
Dice Score Increased 0.046423 -> 0.444213
Saving the model
Epoch: 3,           Training Loss: 0.304153,           Valid Loss: 0.399802,           Dice Score: 0.555236
Dice Score Increased 0.444213 -> 0.555236
Saving the model
Epoch: 4,           Training Loss: 0.221903,           Valid Loss: 0.272348,           Dice Score: 0.691593
Dice Score Increased 0.555236 -> 0.691593
Saving the model
Epoch: 5,           Training Loss: 0.190326,           Valid Loss: 0.169090,           Dice Score: 0.715115
Dice Score Increased 0.691593 -> 0.715115
Saving the model
Epoch: 6,           Training Loss: 0.151642,           Valid Loss: 0.188032,           Dice Score: 0.674143
Epoch: 7,           Training Loss: 0.135309,           V

## Evaluation


In [ ]:
model = UNet().to(device)
model.load_state_dict(torch.load("./saved_model.pth"))
model.eval()

In [11]:
validation_loss, validation_score = validate_epoch(
    model,
    validation_data_loader,
    criterion,
    device,
)
validation_score / len(validation_data_loader)

0.8882506772987331

## Test-set predictions


In [ ]:
model.eval()

for image, filenames in test_data_loader:
    image = image.unsqueeze(1).to(device=device)
    prediction = model(image)

    for index in range(prediction.shape[0]):
        mask = prediction[index].argmax(axis=0).cpu().detach().numpy()
        cv2.imwrite(
            os.path.join("./data/test", "mask", filenames[index] + "_mask.png"),
            mask,
        )